# EDA ICFES Saber 11 – Valle del Cauca
**Fase 3: Análisis Exploratorio y Validación (Python)** · Tareas 16, 17 y 18

| Tarea | Qué responde este notebook | Sección |
|---|---|---|
| 16 | Análisis univariado del puntaje global (estadísticas descriptivas + gráficos de distribución) | 3 |
| 17 | Análisis bivariado y matriz de correlación (mín. 3 hallazgos documentados) | 4 |
| 18 | Validación ética: sesgos, generalizaciones indebidas y lenguaje estigmatizante | 5 |

**Cómo usarlo:** ejecuta las celdas en orden (`Entorno de ejecución → Ejecutar todo`).
El archivo `notas_icfes.csv` pesa ~200 MB; súbelo a `/content` o móntalo desde Google Drive (ver sección 1).

> **Principio guía:** con más de 400 mil registros *todo* resulta "estadísticamente significativo".
> Por eso, para decidir qué variables importan usamos **tamaño del efecto** (|ρ|, ΔR²), no valores‑p.

## 1. Configuración y carga de datos

In [ ]:
import os, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.linear_model import LinearRegression

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams.update({"figure.dpi": 110, "axes.titlesize": 12, "axes.titleweight": "bold"})
pd.set_option("display.max_columns", 60, "display.width", 200, "display.float_format", "{:,.3f}".format)

os.makedirs("figuras", exist_ok=True)
os.makedirs("resultados", exist_ok=True)

def mostrar(nombre):
    """Guarda la figura en /figuras y la muestra."""
    plt.tight_layout()
    plt.savefig(f"figuras/{nombre}.png", dpi=150, bbox_inches="tight")
    plt.show()

In [ ]:
# ---- Ruta del archivo -------------------------------------------------------
# Opción A: archivo subido a Colab (panel izquierdo → carpeta "Archivos" → subir)
RUTA_CSV = "/content/notas_icfes.csv"
# Opción B: Google Drive (descomenta las 2 líneas siguientes y ajusta la ruta)
# from google.colab import drive; drive.mount("/content/drive")
# RUTA_CSV = "/content/drive/MyDrive/notas_icfes.csv"

if not os.path.exists(RUTA_CSV):                       # Opción C: subir en el momento
    from google.colab import files
    print("Selecciona notas_icfes.csv (puede tardar unos minutos)...")
    RUTA_CSV = next(iter(files.upload()))

# 'NAN' viene escrito como TEXTO en el archivo: lo convertimos a nulo real.
df_raw = pd.read_csv(RUTA_CSV, low_memory=False, na_values=["NAN", "NaN", "nan", ""])
print(f"Filas: {df_raw.shape[0]:,} | Columnas: {df_raw.shape[1]}")

## 2. Auditoría de calidad y preparación
Antes de calcular cualquier estadístico hay que saber **qué tan confiable es cada columna**.

In [ ]:
# 2.1 Nulos por columna
nulos = df_raw.isna().sum()
nulos = nulos[nulos > 0].to_frame("nulos")
nulos["%"] = 100 * nulos["nulos"] / len(df_raw)
display(nulos.sort_values("nulos", ascending=False))

In [ ]:
# 2.2 Duplicados exactos (misma fila repetida en las 51 columnas)
n_dup = df_raw.duplicated().sum()
print(f"Filas duplicadas exactas: {n_dup:,} ({100*n_dup/len(df_raw):.1f} %)")
dup_por_anio = df_raw[df_raw.duplicated()].groupby("AÑO").size().rename("duplicadas")
tot_anio = df_raw.groupby("AÑO").size().rename("total")
tabla_dup = pd.concat([tot_anio, dup_por_anio], axis=1).fillna(0).astype(int)
tabla_dup["% dup"] = 100 * tabla_dup["duplicadas"] / tabla_dup["total"]
display(tabla_dup)
# NOTA: ESTU_CONSECUTIVO NO es una llave única en los años 2010-2013 (mismo consecutivo con notas
# distintas), por eso se eliminan solo duplicados EXACTOS de toda la fila, no por consecutivo.

In [ ]:
# 2.3 Composición del dato por año: ¿qué tipo de colegios aparecen en cada año?
comp = (df_raw.drop_duplicates()
        .assign(oficial=lambda d: (d.COLE_NATURALEZA == "OFICIAL").astype(float).where(d.COLE_NATURALEZA.notna()),
                calB=lambda d: (d.COLE_CALENDARIO == "B").astype(float))
        .groupby("AÑO").agg(n=("PUNT_GLOBAL", "size"), pct_oficial=("oficial", "mean"),
                            pct_calendarioB=("calB", "mean"), media_global=("PUNT_GLOBAL", "mean")))
comp[["pct_oficial", "pct_calendarioB"]] *= 100
display(comp)
# Los años 2018, 2020 y 2021 tienen 0 % de colegios oficiales y casi solo calendario B:
# no representan a toda la población estudiantil → los marcamos y hacemos análisis de robustez.
ANIOS_SESGADOS = comp.index[(comp.pct_oficial < 5)].tolist()
print("Años con composición no representativa:", ANIOS_SESGADOS)

In [ ]:
# 2.4 Construcción del dataset analítico
df = df_raw.drop_duplicates().copy()
n_ini = len(df)

# (a) Registros sin respuesta: puntaje global = 0 en todas las áreas (no es un puntaje real)
sin_resp = (df.PUNT_GLOBAL == 0)
df = df[~sin_resp].copy()
print(f"Tras eliminar duplicados: {n_ini:,} | registros con puntaje global = 0 retirados: {sin_resp.sum()}")

# (b) Variables ordinales
EDU = {"NINGUNO": 0, "PRIMARIA INCOMPLETA": 1, "PRIMARIA COMPLETA": 2,
       "SECUNDARIA (BACHILLERATO) INCOMPLETA": 3, "SECUNDARIA (BACHILLERATO) COMPLETA": 4,
       "TECNICA O TECNOLOGICA INCOMPLETA": 5, "TECNICA O TECNOLOGICA COMPLETA": 6,
       "EDUCACION PROFESIONAL INCOMPLETA": 7, "EDUCACION PROFESIONAL COMPLETA": 8, "POSTGRADO": 9}
# 'NO SABE', 'NO APLICA' y valores inválidos ('SI','NO') quedan como nulos.
df["EDU_MADRE"] = df.FAMI_EDUCACIONMADRE.map(EDU)
df["EDU_PADRE"] = df.FAMI_EDUCACIONPADRE.map(EDU)
df["ESTRATO"] = pd.to_numeric(df.FAMI_ESTRATOVIVIENDA.str.extract(r"(\d)")[0])   # 'SIN ESTRATO' → nulo

# (c) Personas en el hogar: el formulario cambió en 2017 (palabras → rangos). Se unifica en 5 grupos.
PERS = {"UNA": 1, "DOS": 1, "TRES": 2, "CUATRO": 2, "CINCO": 3, "SEIS": 3, "SIETE": 4, "OCHO": 4,
        "NUEVE": 5, "DIEZ": 5, "ONCE": 5, "DOCE O MAS": 5,
        "1 A 2": 1, "3 A 4": 2, "5 A 6": 3, "7 A 8": 4, "9 O MAS": 5}     # 1=1-2 … 5=9 o más
df["PERSONAS_HOGAR"] = df.FAMI_PERSONASHOGAR.map(PERS)
df["CUARTOS"] = df.FAMI_CUARTOSHOGAR

# (d) Variables binarias (1 = categoría de interés)
si_no = {"SI": 1, "NO": 0}
df["TIENE_AUTOMOVIL"] = df.FAMI_TIENEAUTOMOVIL.map(si_no)
df["TIENE_COMPUTADOR"] = df.FAMI_TIENECOMPUTADOR.map(si_no)
df["TIENE_INTERNET"] = df.FAMI_TIENEINTERNET.map(si_no)
df["TIENE_LAVADORA"] = df.FAMI_TIENELAVADORA.map(si_no)
df["COLEGIO_NO_OFICIAL"] = df.COLE_NATURALEZA.map({"NO OFICIAL": 1, "OFICIAL": 0})
df["COLEGIO_RURAL"] = df.COLE_AREA_UBICACION.map({"RURAL": 1, "URBANO": 0})
df["COLEGIO_BILINGUE"] = df.COLE_BILINGUE.map({"S": 1, "N": 0})
df["CALENDARIO_B"] = df.COLE_CALENDARIO.map({"B": 1, "A": 0})           # 'OTRO' → nulo
df["HOMBRE"] = df.ESTU_GENERO.map({"M": 1, "F": 0})

# (e) Edad al momento de la prueba (se descartan valores imposibles)
nac = pd.to_datetime(df.ESTU_FECHANACIMIENTO, errors="coerce")
edad = (pd.to_datetime(df["AÑO"].astype(str) + "-06-30") - nac).dt.days / 365.25
df["EDAD"] = edad.where(edad.between(14, 25))

# (f) Marca de composición y periodo
df["ANIO_SESGADO"] = df["AÑO"].isin(ANIOS_SESGADOS)
print(f"Dataset analítico: {len(df):,} filas")

## 3. TAREA 16 – Análisis univariado del puntaje global
**Entregable:** estadísticas descriptivas + gráficos de distribución del puntaje global.

In [ ]:
# 3.1 Estadísticas descriptivas del puntaje global y de las 5 áreas
AREAS = ["PUNT_LECTURA_CRITICA", "PUNT_MATEMATICAS", "PUNT_SOCIALES_CIUDADANAS", "PUNT_C_NATURALES", "PUNT_INGLES"]
PUNT = ["PUNT_GLOBAL"] + AREAS

def describir(s):
    q1, q3 = s.quantile([.25, .75])
    return pd.Series({"n": s.count(), "media": s.mean(), "mediana": s.median(), "desv_est": s.std(),
                      "min": s.min(), "P5": s.quantile(.05), "Q1": q1, "Q3": q3, "P95": s.quantile(.95),
                      "max": s.max(), "CV_%": 100 * s.std() / s.mean(), "asimetría": stats.skew(s),
                      "curtosis_exceso": stats.kurtosis(s)})
desc = df[PUNT].apply(describir).T
display(desc)
desc.to_csv("resultados/t16_descriptivas.csv")

g = df.PUNT_GLOBAL
q1, q3 = g.quantile([.25, .75]); iqr = q3 - q1
lim_inf, lim_sup = q1 - 1.5 * iqr, q3 + 1.5 * iqr
n_out = ((g < lim_inf) | (g > lim_sup)).sum()
print(f"Rango intercuartílico = {iqr:.0f} | límites de outliers (1.5·IQR): [{lim_inf:.0f}, {lim_sup:.0f}]")
print(f"Outliers: {n_out:,} ({100*n_out/len(g):.2f} %) → son puntajes válidos (extremos reales), NO se eliminan.")

In [ ]:
# 3.2 Distribución del puntaje global: histograma + boxplot + Q-Q
fig, ax = plt.subplots(1, 3, figsize=(16, 4.3), gridspec_kw={"width_ratios": [2.2, 1, 1.3]})
sns.histplot(g, bins=50, kde=True, color="#2a6f97", ax=ax[0])
ax[0].axvline(g.mean(), color="#c1121f", ls="--", label=f"Media {g.mean():.1f}")
ax[0].axvline(g.median(), color="#2d6a4f", ls=":", label=f"Mediana {g.median():.0f}")
ax[0].set(title="Distribución del puntaje global", xlabel="Puntaje global (0–500)", ylabel="Estudiantes"); ax[0].legend()
sns.boxplot(y=g, color="#8ecae6", ax=ax[1], fliersize=1.5); ax[1].set(title="Boxplot", ylabel="")
muestra = g.sample(20000, random_state=1)
stats.probplot(muestra, dist="norm", plot=ax[2]); ax[2].set_title("Q-Q vs. normal (muestra 20 mil)")
ax[2].set(xlabel="Cuantiles teóricos (normal)", ylabel="Cuantiles observados")
ax[2].get_lines()[0].set(markersize=2, alpha=.4)
mostrar("t16_global_distribucion")
print(f"Asimetría = {stats.skew(g):.2f} | curtosis en exceso = {stats.kurtosis(g):.2f} → "
      "forma aproximadamente normal con ligera cola a la derecha.")

In [ ]:
# 3.3 Distribución de las cinco áreas
fig, axes = plt.subplots(1, 5, figsize=(18, 3.4), sharey=False)
etiquetas = ["Lectura crítica", "Matemáticas", "Sociales y ciudad.", "Ciencias nat.", "Inglés"]
for a, col, et in zip(axes, AREAS, etiquetas):
    sns.histplot(df[col], bins=40, color="#219ebc", ax=a)
    a.axvline(df[col].mean(), color="#c1121f", ls="--", lw=1)
    a.set(title=f"{et}\nμ={df[col].mean():.1f}  σ={df[col].std():.1f}", xlabel="Puntaje (0–100)", ylabel="")
mostrar("t16_areas_distribucion")

In [ ]:
# 3.4 Percentiles y evolución por año (con tamaño de muestra y composición)
pct = g.quantile([.05, .10, .25, .50, .75, .90, .95]).rename(lambda p: f"P{int(p*100)}").to_frame("puntaje global")
display(pct.T)

anual = df.groupby("AÑO").agg(n=("PUNT_GLOBAL", "size"), media=("PUNT_GLOBAL", "mean"),
                              mediana=("PUNT_GLOBAL", "median"), desv=("PUNT_GLOBAL", "std"))
fig, ax = plt.subplots(figsize=(11, 4))
ax.bar(anual.index, anual.n, color="#dee2e6")
ax.set_ylabel("n estudiantes (barras)"); ax.grid(False)
ax2 = ax.twinx(); ax2.grid(False)
sesgado = anual.index.isin(ANIOS_SESGADOS)
ax2.plot(anual.index, anual.media.where(~sesgado), color="#1d3557", marker="o", label="Años representativos")
ax2.scatter(anual.index[sesgado], anual.media[sesgado], color="#c1121f", s=70, zorder=3, label="Años con composición no representativa")
ax2.set(ylabel="Media puntaje global", title="Media anual del puntaje global"); ax2.legend(loc="upper left", fontsize=9)
ax.set_xticks(anual.index)
mostrar("t16_evolucion_anual")
display(anual)

In [ ]:
# 3.5 Perfil univariado de las variables sociodemográficas (para saber qué cubre la muestra)
fig, axes = plt.subplots(2, 4, figsize=(18, 7))
def barras(ax, serie, titulo, orden=None, rot=0):
    vc = serie.value_counts(normalize=True, dropna=True).mul(100)
    if orden is not None: vc = vc.reindex(orden)
    sns.barplot(x=vc.index.astype(str), y=vc.values, color="#457b9d", ax=ax)
    ax.set(title=titulo, xlabel="", ylabel="%"); ax.tick_params(axis="x", rotation=rot)
barras(axes[0, 0], df.ESTRATO, "Estrato", orden=[1, 2, 3, 4, 5, 6])
barras(axes[0, 1], df.EDU_MADRE, "Educación madre (0=ninguno … 9=posgrado)", orden=list(range(10)))
barras(axes[0, 2], df.EDU_PADRE, "Educación padre (0=ninguno … 9=posgrado)", orden=list(range(10)))
barras(axes[0, 3], df.PERSONAS_HOGAR, "Personas en hogar (1=1–2 … 5=9+)", orden=[1, 2, 3, 4, 5])
barras(axes[1, 0], df.COLE_NATURALEZA, "Naturaleza del colegio")
barras(axes[1, 1], df.COLE_AREA_UBICACION, "Zona del colegio")
barras(axes[1, 2], df.ESTU_GENERO.map({"F": "Femenino", "M": "Masculino"}), "Género")
barras(axes[1, 3], df.COLE_JORNADA, "Jornada", rot=45)
mostrar("t16_perfil_sociodemografico")

## 4. TAREA 17 – Análisis bivariado y matriz de correlación
### 4.1 ¿Qué variables entran a la matriz? (selección razonada)
No todas las columnas sirven para una matriz de correlación. Se aplicaron 5 filtros:

| Filtro | Qué se excluye | Por qué |
|---|---|---|
| 1. Fuga de información | `PUNT_*` por área y `DESEMP_INGLES` | Son componentes/derivados del puntaje global: correlan por construcción, no por "factor" |
| 2. Identificadores y códigos | `ESTU_CONSECUTIVO`, códigos DANE/ICFES, nombres de colegio | No tienen significado numérico |
| 3. Sin variabilidad | Depto./país (`VALLE`, `COLOMBIA` > 99 %), `ESTU_PRIVADO_LIBERTAD` | Casi constantes → correlación inestable |
| 4. Calidad | Variables con muchos nulos o cambios de formulario sin armonizar | Se armonizaron (p. ej. personas en hogar) o se descartaron |
| 5. Tipo de variable | Nominales sin orden (jornada, carácter, municipio) | Pearson/Spearman no aplican → se miden aparte con **η²** |

Quedan **variables numéricas, ordinales y binarias** con relevancia teórica: nivel socioeconómico del hogar,
capital educativo de los padres, acceso a recursos, contexto del colegio y características del estudiante.

In [ ]:
# 4.2 Definición de variables candidatas
CANDIDATAS = {
    "ESTRATO": "Estrato de la vivienda (1–6)",
    "EDU_MADRE": "Nivel educativo de la madre (0–9)",
    "EDU_PADRE": "Nivel educativo del padre (0–9)",
    "CUARTOS": "Cuartos en el hogar",
    "PERSONAS_HOGAR": "Personas en el hogar (grupos 1–5)",
    "TIENE_COMPUTADOR": "Tiene computador (0/1)",
    "TIENE_INTERNET": "Tiene internet (0/1)",
    "TIENE_AUTOMOVIL": "Tiene automóvil (0/1)",
    "TIENE_LAVADORA": "Tiene lavadora (0/1)",
    "COLEGIO_NO_OFICIAL": "Colegio no oficial/privado (0/1)",
    "COLEGIO_RURAL": "Colegio en zona rural (0/1)",
    "COLEGIO_BILINGUE": "Colegio bilingüe (0/1)",
    "CALENDARIO_B": "Colegio calendario B (0/1)",
    "HOMBRE": "Estudiante hombre (0/1)",
    "EDAD": "Edad al presentar la prueba",
}
VARS = list(CANDIDATAS)
cobertura = pd.DataFrame({"descripción": pd.Series(CANDIDATAS),
                          "% con dato": (100 * df[VARS].notna().mean()).round(1),
                          "valores únicos": df[VARS].nunique()})
display(cobertura)

In [ ]:
# 4.3 Relación bivariada visual: puntaje global según las variables principales
fig, axes = plt.subplots(2, 3, figsize=(17, 8.5))
plot_df = df.assign(ESTRATO_=df.ESTRATO.astype("Int64").astype(str), EDU_MADRE_=df.EDU_MADRE.astype("Int64").astype(str),
                    COMPUTADOR_=df.TIENE_COMPUTADOR.map({1: "Sí", 0: "No"}),
                    GENERO_=df.ESTU_GENERO.map({"F": "Femenino", "M": "Masculino"}))
def caja(ax, x, titulo, orden=None):
    d = plot_df.dropna(subset=[x]); d = d[d[x] != "<NA>"]
    sns.boxplot(data=d, x=x, y="PUNT_GLOBAL", order=orden, color="#8ecae6", fliersize=1, ax=ax)
    ax.set(title=titulo, xlabel="", ylabel="Puntaje global")
caja(axes[0, 0], "ESTRATO_", "Puntaje global por estrato", [str(i) for i in range(1, 7)])
caja(axes[0, 1], "EDU_MADRE_", "Por educación de la madre (0=ninguno … 9=posgrado)", [str(i) for i in range(10)])
caja(axes[0, 2], "COLE_NATURALEZA", "Por naturaleza del colegio")
caja(axes[1, 0], "COMPUTADOR_", "Por tenencia de computador", ["No", "Sí"])
caja(axes[1, 1], "COLE_AREA_UBICACION", "Por zona del colegio")
caja(axes[1, 2], "GENERO_", "Por género")
mostrar("t17_boxplots_bivariados")

In [ ]:
# 4.4 MATRIZ DE CORRELACIÓN (Spearman: válida para ordinales y binarias, robusta a outliers)
corr = df[["PUNT_GLOBAL"] + VARS].corr(method="spearman")
orden_vars = corr["PUNT_GLOBAL"].drop("PUNT_GLOBAL").abs().sort_values(ascending=False).index.tolist()
corr = corr.loc[["PUNT_GLOBAL"] + orden_vars, ["PUNT_GLOBAL"] + orden_vars]      # ordenada de mayor a menor |ρ| con el puntaje
mascara = np.triu(np.ones_like(corr, dtype=bool), k=1)

fig, ax = plt.subplots(figsize=(13, 10.5))
sns.heatmap(corr, mask=mascara, annot=True, fmt=".2f", cmap="RdBu_r", center=0, vmin=-1, vmax=1,
            linewidths=.5, cbar_kws={"label": "ρ de Spearman", "shrink": .8}, annot_kws={"size": 8.5}, ax=ax)
ax.grid(False); ax.set_title("Matriz de correlación de Spearman – Puntaje global y variables sociodemográficas\n(ordenada por magnitud de la correlación con el puntaje global)")
mostrar("t17_matriz_correlacion")
corr.to_csv("resultados/t17_matriz_spearman.csv")

In [ ]:
# 4.5 Ranking de asociación con el puntaje global (con IC 95 % y tamaño del efecto)
def clasificar(r):
    r = abs(r)
    return "Alta" if r >= .5 else "Moderada" if r >= .3 else "Baja" if r >= .1 else "Despreciable"

filas = []
for v in VARS:
    par = df[["PUNT_GLOBAL", v]].dropna()
    rho = par.corr(method="spearman").iloc[0, 1]
    pear = par.corr(method="pearson").iloc[0, 1]
    n = len(par)
    se = np.sqrt((1 + rho**2 / 2) / (n - 3))                 # error estándar de Bonett–Wright
    z = np.arctanh(rho)
    filas.append({"variable": v, "descripción": CANDIDATAS[v], "n": n, "rho_spearman": rho, "pearson": pear,
                  "IC95_inf": np.tanh(z - 1.96 * se), "IC95_sup": np.tanh(z + 1.96 * se),
                  "|rho|": abs(rho), "magnitud": clasificar(rho)})
rank = pd.DataFrame(filas).sort_values("|rho|", ascending=False).reset_index(drop=True)
display(rank)
rank.to_csv("resultados/t17_ranking_correlaciones.csv", index=False)

fig, ax = plt.subplots(figsize=(9, 6))
orden = rank.sort_values("rho_spearman")
ax.barh(orden.variable, orden.rho_spearman, color=np.where(orden.rho_spearman > 0, "#2a6f97", "#c1121f"))
ax.axvline(0, color="k", lw=.8)
for x in (-.1, .1): ax.axvline(x, color="gray", ls=":", lw=1)
ax.set(title="Correlación de Spearman con el puntaje global\n(líneas punteadas: umbral de efecto bajo |ρ| = 0.10)", xlabel="ρ de Spearman")
mostrar("t17_ranking_barras")

In [ ]:
# 4.6 ¿Qué áreas son más sensibles al contexto? (predictores × 5 áreas + global)
corr_areas = pd.DataFrame({p: df[VARS].corrwith(df[p], method="spearman") for p in PUNT})
corr_areas.columns = ["Global", "Lectura crítica", "Matemáticas", "Sociales y ciudadanas", "Ciencias naturales", "Inglés"]
fig, ax = plt.subplots(figsize=(9, 7.5))
sns.heatmap(corr_areas.loc[rank.variable], annot=True, fmt=".2f", cmap="RdBu_r", center=0, vmin=-.7, vmax=.7,
            linewidths=.5, cbar_kws={"label": "ρ de Spearman"}, ax=ax)
ax.grid(False); ax.set_title("Correlación de cada variable con el puntaje global y con cada área")
mostrar("t17_correlacion_por_area")
corr_areas.to_csv("resultados/t17_correlacion_por_area.csv")

In [ ]:
# 4.7 Variables nominales (sin orden): razón de correlación η² = varianza del puntaje explicada por el grupo
def eta2(cat, y="PUNT_GLOBAL", min_n=30):
    d = df[[cat, y]].dropna()
    vc = d[cat].value_counts(); d = d[d[cat].isin(vc[vc >= min_n].index)]        # grupos con al menos 30 casos
    gm = d[y].mean()
    ss_b = d.groupby(cat)[y].agg(lambda s: len(s) * (s.mean() - gm) ** 2).sum()
    ss_t = ((d[y] - gm) ** 2).sum()
    return ss_b / ss_t, d[cat].nunique(), len(d)

nominales = {"COLE_JORNADA": "Jornada", "COLE_CARACTER": "Carácter del colegio",
             "COLE_MCPIO_UBICACION": "Municipio del colegio", "COLE_GENERO": "Tipo de colegio por género",
             "ESTU_TIPODOCUMENTO": "Tipo de documento", "AÑO": "Año de presentación"}
res = []
for c, nom in nominales.items():
    e, k, n = eta2(c)
    res.append({"variable": c, "descripción": nom, "categorías": k, "n": n, "η²": e, "magnitud": clasificar(np.sqrt(e))})
eta = pd.DataFrame(res).sort_values("η²", ascending=False)
display(eta)
eta.to_csv("resultados/t17_eta2_nominales.csv", index=False)
print("Lectura: η² = proporción de la variación del puntaje que 'explica' pertenecer a cada grupo (0–1). "
      "Se calcula solo con grupos de ≥30 casos.")

In [ ]:
# 4.8 Multicolinealidad entre predictores (¿qué variables miden lo mismo?)
cp = df[VARS].corr(method="spearman").abs()
pares = (cp.where(np.triu(np.ones(cp.shape), k=1).astype(bool)).stack()
           .sort_values(ascending=False).rename("|ρ|").reset_index())
pares.columns = ["var_1", "var_2", "|ρ|"]
display(pares.head(10))

# VIF calculado sin statsmodels: VIF_j = 1 / (1 − R²_j), regresando cada variable contra las demás
sub = df[VARS].dropna()
Z = (sub - sub.mean()) / sub.std()
vif = {}
for v in VARS:
    otros = [c for c in VARS if c != v]
    r2 = LinearRegression().fit(Z[otros], Z[v]).score(Z[otros], Z[v])
    vif[v] = 1 / (1 - r2)
vif = pd.Series(vif, name="VIF").sort_values(ascending=False)
display(vif.to_frame().T)
print("Regla práctica: VIF > 5 = colinealidad relevante; VIF > 10 = severa.")
vif.to_csv("resultados/t17_vif.csv")

In [ ]:
# 4.9 Contribución conjunta: ¿qué variables conservan su aporte al controlar por las demás?
y = sub.join(df["PUNT_GLOBAL"], how="left")["PUNT_GLOBAL"]
yz = (y - y.mean()) / y.std()
full = LinearRegression().fit(Z, yz)
r2_full = full.score(Z, yz)
aporte = []
for v in VARS:
    otros = [c for c in VARS if c != v]
    r2_sin = LinearRegression().fit(Z[otros], yz).score(Z[otros], yz)
    aporte.append({"variable": v, "beta_estandarizado": full.coef_[VARS.index(v)], "ΔR²_único": r2_full - r2_sin})
aporte = pd.DataFrame(aporte).sort_values("ΔR²_único", ascending=False).reset_index(drop=True)
display(aporte)
print(f"R² del modelo con las {len(VARS)} variables: {r2_full:.3f}  (n = {len(sub):,})")

# Mismo modelo pero controlando por año (efectos fijos)
Zy = pd.concat([Z, pd.get_dummies(df.loc[sub.index, "AÑO"], prefix="a", drop_first=True).astype(float)], axis=1)
r2_anio = LinearRegression().fit(Zy, yz).score(Zy, yz)
print(f"R² controlando además por año: {r2_anio:.3f}")
aporte.to_csv("resultados/t17_aporte_unico.csv", index=False)

In [ ]:
# 4.10 Robustez: ¿los resultados cambian si excluimos años no representativos o la prueba anterior a 2014?
df["GLOBAL_Z_ANIO"] = df.groupby("AÑO").PUNT_GLOBAL.transform(lambda x: (x - x.mean()) / x.std())   # puntaje relativo a su año
subconjuntos = {"Todos\n(sin duplicados)": ("PUNT_GLOBAL", df),
                "Sin años\nsesgados": ("PUNT_GLOBAL", df[~df.ANIO_SESGADO]),
                "Desde 2014 y sin\naños sesgados": ("PUNT_GLOBAL", df[(~df.ANIO_SESGADO) & (df["AÑO"] >= 2014)]),
                "Puntaje centrado\npor año": ("GLOBAL_Z_ANIO", df),
                "Solo colegios\noficiales": ("PUNT_GLOBAL", df[df.COLE_NATURALEZA == "OFICIAL"])}
rob = pd.DataFrame({k: d[VARS].corrwith(d[yv], method="spearman") for k, (yv, d) in subconjuntos.items()})
rob = rob.loc[rank.variable]
display(rob)
rob.to_csv("resultados/t17_robustez.csv")
fig, ax = plt.subplots(figsize=(11, 6.5))
sns.heatmap(rob, annot=True, fmt=".2f", cmap="RdBu_r", center=0, vmin=-.7, vmax=.7, linewidths=.5, ax=ax,
            cbar_kws={"label": "ρ de Spearman con puntaje global"})
ax.grid(False); ax.set_title("Robustez de las correlaciones según el subconjunto de datos\n(casilla vacía = variable sin variación en ese subconjunto)")
plt.xticks(rotation=0)
mostrar("t17_robustez")

### 4.11 De la correlación a los puntos: brechas y superposición entre grupos
Un ρ = 0.33 es abstracto. Aquí se traduce a **puntos del puntaje global** y a **cuánto se traslapan los grupos**
(`P(A>B)` = probabilidad de que un estudiante al azar del grupo A supere a uno del grupo B; 50 % = sin diferencia;
`d_Cohen`: 0.2 = pequeño, 0.5 = mediano, 0.8 = grande).

In [ ]:
def prob_sup(a, b):
    """P(A>B): probabilidad de superioridad (empates = 0.5), calculada con rangos."""
    r = stats.rankdata(pd.concat([a, b])); n1, n2 = len(a), len(b)
    return (r[:n1].sum() - n1 * (n1 + 1) / 2) / (n1 * n2)

G = df.PUNT_GLOBAL
comparaciones = {
    "Estrato 6 vs. estrato 1": (df.ESTRATO == 6, df.ESTRATO == 1),
    "Estratos 4–6 vs. estratos 1–2": (df.ESTRATO >= 4, df.ESTRATO <= 2),
    "Madre profesional/posgrado vs. madre hasta primaria": (df.EDU_MADRE >= 7, df.EDU_MADRE <= 2),
    "Con internet vs. sin internet": (df.TIENE_INTERNET == 1, df.TIENE_INTERNET == 0),
    "Colegio no oficial vs. oficial": (df.COLEGIO_NO_OFICIAL == 1, df.COLEGIO_NO_OFICIAL == 0),
    "Hombres vs. mujeres": (df.HOMBRE == 1, df.HOMBRE == 0),
    "Colegio urbano vs. rural": (df.COLEGIO_RURAL == 0, df.COLEGIO_RURAL == 1),
}
filas = []
for nom, (ma, mb) in comparaciones.items():
    a, b = G[ma], G[mb]
    sp = np.sqrt(((len(a) - 1) * a.var() + (len(b) - 1) * b.var()) / (len(a) + len(b) - 2))
    filas.append({"comparación (A vs. B)": nom, "n_A": len(a), "n_B": len(b), "media_A": a.mean(), "media_B": b.mean(),
                  "brecha_puntos": a.mean() - b.mean(), "d_Cohen": (a.mean() - b.mean()) / sp,
                  "P(A>B)": prob_sup(a, b), "% de A sobre la media general": 100 * (a > G.mean()).mean(),
                  "% de B sobre la media general": 100 * (b > G.mean()).mean()})
brechas = pd.DataFrame(filas)
display(brechas)
brechas.to_csv("resultados/t17_brechas.csv", index=False)

# Edad y jornada: la edad captura sobre todo a estudiantes de jornada nocturna/sabatina (población adulta trabajadora)
df["GRUPO_EDAD"] = pd.cut(np.floor(df.EDAD), [0, 16, 17, 18, 20, 99], labels=["≤16", "17", "18", "19–20", "21+"])
por_edad = df.groupby("GRUPO_EDAD", observed=True).PUNT_GLOBAL.agg(n="size", media="mean", mediana="median")
por_jornada = df.groupby("COLE_JORNADA").agg(n=("PUNT_GLOBAL", "size"), media_global=("PUNT_GLOBAL", "mean"),
                                             edad_media=("EDAD", "mean")).sort_values("media_global")
display(por_edad); display(por_jornada)
por_edad.to_csv("resultados/t17_por_edad.csv"); por_jornada.to_csv("resultados/t17_por_jornada.csv")

### 4.12 Tabla de decisión: ¿qué variables son las más relevantes para la matriz?
Se clasifican con reglas explícitas (para que la decisión sea auditable):
* **NÚCLEO**: |ρ| ≥ 0.30 con el puntaje global (efecto moderado o mayor).
* **COMPLEMENTARIA**: 0.10 ≤ |ρ| < 0.30 (efecto bajo pero real).
* **EFECTO BAJO**: |ρ| < 0.10 → se mantiene en el informe *por equidad* (género, zona), pero **no** se presenta como factor de impacto.

Además se revisan: estabilidad del signo entre subconjuntos, si persiste dentro de colegios oficiales,
redundancia con otra variable (|ρ| entre predictores) y aporte único (ΔR²).

In [ ]:
r_ = rank.set_index("variable")
sub_ok = rob.iloc[:, :4]                                  # subconjuntos con variación en todas las variables
red_max = cp.where(~np.eye(len(cp), dtype=bool)).max(axis=1)
red_con = cp.where(~np.eye(len(cp), dtype=bool)).idxmax(axis=1)
ofi = rob.iloc[:, 4]
decision = pd.DataFrame(index=r_.index)
decision["ρ (todos)"] = r_["rho_spearman"]
decision["ρ mín–máx en robustez"] = [f"{sub_ok.loc[v].min():+.2f} a {sub_ok.loc[v].max():+.2f}" for v in decision.index]
decision["signo estable"] = [bool(np.sign(sub_ok.loc[v]).nunique() == 1) for v in decision.index]
decision["persiste en oficiales (|ρ|≥.10)"] = [("n/a" if pd.isna(ofi[v]) else ("sí" if abs(ofi[v]) >= .10 else "no")) for v in decision.index]
decision["ΔR² único"] = aporte.set_index("variable")["ΔR²_único"]
decision["VIF"] = vif
decision["redundante con (|ρ|)"] = [f"{red_con[v]} ({red_max[v]:.2f})" if red_max[v] >= .60 else "—" for v in decision.index]
decision["decisión"] = ["NÚCLEO" if abs(r_.loc[v, "rho_spearman"]) >= .30 else
                        "COMPLEMENTARIA" if abs(r_.loc[v, "rho_spearman"]) >= .10 else "EFECTO BAJO" for v in decision.index]
display(decision)
decision.to_csv("resultados/t17_tabla_decision.csv")

# Matriz reducida para el informe: solo variables con |ρ| ≥ 0.10
VARS_RED = [v for v in r_.index if abs(r_.loc[v, "rho_spearman"]) >= .10]
corr_red = df[["PUNT_GLOBAL"] + VARS_RED].corr(method="spearman")
fig, ax = plt.subplots(figsize=(11, 9))
sns.heatmap(corr_red, mask=np.triu(np.ones_like(corr_red, dtype=bool), k=1), annot=True, fmt=".2f", cmap="RdBu_r",
            center=0, vmin=-1, vmax=1, linewidths=.5, cbar_kws={"label": "ρ de Spearman", "shrink": .8}, ax=ax)
ax.grid(False); ax.set_title(f"Matriz reducida: puntaje global y las {len(VARS_RED)} variables con |ρ| ≥ 0.10")
mostrar("t17_matriz_reducida")

### 4.13 Hallazgos (se generan automáticamente con los resultados de arriba)

In [ ]:
nucleo = decision.index[decision["decisión"] == "NÚCLEO"].tolist()
nombre = lambda v: CANDIDATAS[v].split(" (")[0].lower()
h = []
h.append("H1. Tres variables de contexto familiar concentran la mayor asociación con el puntaje global: "
         + "; ".join(f"{nombre(v)} (ρ={r_.loc[v,'rho_spearman']:+.2f})" for v in nucleo)
         + f". La asociación se mantiene en los cuatro subconjuntos de robustez (ρ entre {sub_ok.loc[nucleo].min().min():+.2f} y {sub_ok.loc[nucleo].max().max():+.2f}). "
           "Es una asociación de magnitud moderada, no una relación determinista.")
areas_soc = corr_areas.loc[nucleo].drop(columns="Global").mean()
h.append(f"H2. El acceso a internet en el hogar (ρ={r_.loc['TIENE_INTERNET','rho_spearman']:+.2f}) y el calendario B "
         f"(ρ={r_.loc['CALENDARIO_B','rho_spearman']:+.2f}) le siguen en magnitud a esos tres. El patrón es similar en las cinco áreas: la asociación "
         f"con el contexto socioeconómico es algo mayor en {areas_soc.idxmax()} (ρ medio {areas_soc.max():.2f}) y algo menor en {areas_soc.idxmin()} (ρ medio {areas_soc.min():.2f}).")
b = brechas.set_index("comparación (A vs. B)")
e61, hm, rr = b.loc["Estrato 6 vs. estrato 1"], b.loc["Hombres vs. mujeres"], b.loc["Colegio urbano vs. rural"]
h.append(f"H3. La mayor brecha observada (estrato 6 vs. estrato 1) es de {e61.brecha_puntos:.0f} puntos en promedio (d de Cohen={e61.d_Cohen:.1f}), "
         f"pero incluso ahí hay dispersión individual: el {e61['% de B sobre la media general']:.0f} % de estudiantes de estrato 1 supera la media general y "
         f"en {100*(1-e61['P(A>B)']):.0f} de cada 100 parejas aleatorias el de estrato 1 obtiene más puntaje que el de estrato 6. "
         f"Para género la brecha promedio es de {hm.brecha_puntos:.0f} puntos (P(A>B)={hm['P(A>B)']:.2f}) y para zona urbana vs. rural de {rr.brecha_puntos:.0f} puntos: "
         "diferencias pequeñas frente a la variación entre individuos del mismo grupo.")
g_area = corr_areas.loc["HOMBRE"].drop("Global")
h.append(f"H4. La asociación con género es de magnitud despreciable (ρ={r_.loc['HOMBRE','rho_spearman']:+.3f}) y varía por área: "
         f"mayor en {g_area.idxmax()} (ρ={g_area.max():+.2f}) y menor en {g_area.idxmin()} (ρ={g_area.min():+.2f}).")
h.append(f"H5. Las variables del hogar se solapan entre sí (mayor |ρ| entre predictores = {pares.iloc[0,2]:.2f}: {pares.iloc[0,0]}–{pares.iloc[0,1]}); "
         f"por eso su efecto no es aditivo. El modelo conjunto explica R²={r2_full:.2f} ({r2_anio:.2f} al controlar por año): "
         f"entre {100*(1-r2_anio):.0f} % y {100*(1-r2_full):.0f} % de la variación individual no se explica por estas variables.")
for t in h: print(t, "\n")

## 5. TAREA 18 – Validación ética
**Objetivo:** garantizar neutralidad y evitar estigmatización de sectores.
Se combinan **verificaciones automáticas** (datos) y una **revisión del lenguaje** (texto).

In [ ]:
# 5.1 Verificación automática A: representatividad y tamaño mínimo de grupos
print("A) Tamaño de grupos usados en comparaciones (mínimo recomendado: 30)")
for c in ["ESTRATO", "EDU_MADRE", "COLE_JORNADA", "COLE_CARACTER", "COLE_AREA_UBICACION", "ESTU_GENERO"]:
    vc = df[c].value_counts()
    pequenos = vc[vc < 30]
    print(f"  {c:22s} categorías={len(vc):2d} | grupo más pequeño n={vc.min():,} | grupos < 30: {len(pequenos)}")

print("\nB) Composición por año (años con 0 % de colegios oficiales no deben usarse para hablar de 'todo el Valle'):")
display(comp.loc[ANIOS_SESGADOS])

In [ ]:
# 5.2 Verificación automática C: datos faltantes por grupo (¿los nulos afectan más a algunos sectores?)
falt = pd.DataFrame({
    "% sin dato educación padre": df.groupby("COLE_NATURALEZA").EDU_PADRE.apply(lambda s: 100 * s.isna().mean()),
    "% sin dato estrato": df.groupby("COLE_NATURALEZA").ESTRATO.apply(lambda s: 100 * s.isna().mean()),
    "% sin dato computador": df.groupby("COLE_NATURALEZA").TIENE_COMPUTADOR.apply(lambda s: 100 * s.isna().mean())})
display(falt)
print("Si un sector concentra más nulos, sus correlaciones son menos confiables: debe señalarse en el informe.")

In [ ]:
# 5.3 Revisión automática de lenguaje estigmatizante / causal en los textos del informe
FRASES_A_EVITAR = {
    r"\bmejores? (colegios?|estudiantes?|alumnos?)\b": "usa 'colegios con puntajes promedio más altos'",
    r"\bpeores? (colegios?|estudiantes?|alumnos?)\b": "usa 'puntajes promedio más bajos'",
    r"\b(determina|causa|provoca|explica por completo)\b": "usa 'se asocia con' (correlación ≠ causalidad)",
    r"\b(los pobres|los ricos|los de estrato)\b": "usa 'estudiantes de hogares de estrato X'",
    r"\b(atrasad[oa]s?|deficientes?|incapaces?|inferiores?)\b": "evita juicios de valor sobre personas o sectores",
    r"\b(las mujeres son|los hombres son|las mujeres no|los hombres no)\b": "no generalices por género",
    r"\b(colegios? rurales? (son|tienen peor))\b": "describe la brecha sin atribuir culpa al sector",
    r"\b(siempre|nunca|todos los|ningún)\b": "evita absolutos: hay variación dentro de cada grupo",
}
import re
def revisar_lenguaje(texto):
    alertas = [(pat, sug) for pat, sug in FRASES_A_EVITAR.items() if re.search(pat, texto.lower())]
    return alertas

todo = "\n".join(h)
alertas = revisar_lenguaje(todo)
print("Alertas de lenguaje en los hallazgos H1–H5:", "ninguna ✔" if not alertas else alertas)

### 5.4 Checklist ético (aplicar y marcar en el informe Word)
| # | Criterio | Cómo se verificó en este notebook |
|---|---|---|
| 1 | Datos personales protegidos | No se muestran documentos, nombres ni fechas de nacimiento individuales; solo agregados |
| 2 | Representatividad | Sección 2.3 y 5.1-B: años con composición no representativa marcados y con análisis de robustez |
| 3 | Calidad de datos documentada | Sección 2: nulos, duplicados, categorías inválidas y cambio de formulario |
| 4 | Correlación ≠ causalidad | Todas las conclusiones usan "se asocia con"; revisión automática en 5.3 |
| 5 | Tamaño del efecto sobre significancia | Se reportan |ρ|, ΔR² e IC 95 %, no valores‑p |
| 6 | Sin fuga de información | `PUNT_*` por área y `DESEMP_INGLES` excluidos como "factores" |
| 7 | Grupos pequeños | Sección 5.1-A: mínimo de 30 casos por grupo |
| 8 | Variación intragrupo | Los boxplots muestran la superposición entre grupos: ningún grupo es homogéneo |
| 9 | Lenguaje neutral | Revisión 5.3 + guía de redacción del informe |
| 10 | Variables sensibles (género, estrato, zona) | Se reportan como contexto, sin atribuir mérito ni deficiencia a personas o sectores |

## 6. Exportar resultados
Las tablas quedan en `resultados/` y las figuras en `figuras/`. Descárgalas en un ZIP para anexarlas al Word.

In [ ]:
import shutil
shutil.make_archive("resultados_eda_icfes", "zip", ".", "resultados")
shutil.make_archive("figuras_eda_icfes", "zip", ".", "figuras")
try:
    from google.colab import files
    files.download("resultados_eda_icfes.zip"); files.download("figuras_eda_icfes.zip")
except ImportError:
    print("Archivos generados: resultados_eda_icfes.zip y figuras_eda_icfes.zip")